# Task 1: Data Extraction, EDA & Risk Metrics

**GMF Investments — Portfolio Management Optimization**

Assets: TSLA (high-risk), BND (low-risk), SPY (moderate)

Period: January 1, 2015 – June 30, 2026

**Author:** Sosina Ayele

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded!')

## 1. Extract Financial Data

In [ ]:
TICKERS = ['TSLA', 'BND', 'SPY']
START = '2015-01-01'
END   = '2026-06-30'

print(f'Fetching data for {TICKERS} from {START} to {END}...')
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True)
print(f'Raw shape: {raw.shape}')
print(raw.tail(3))

In [ ]:
# Extract Close prices
prices = raw['Close'].copy()
prices.index = pd.to_datetime(prices.index)
prices.columns.name = None

print('=== Missing Values ===')
print(prices.isnull().sum())

# Forward fill small gaps (weekends/holidays)
prices = prices.ffill().bfill()
print(f'\nAfter fill — missing: {prices.isnull().sum().sum()}')
print(f'Date range: {prices.index[0].date()} to {prices.index[-1].date()}')
print(f'Trading days: {len(prices):,}')
print(f'\nDescriptive Stats:')
print(prices.describe().round(2))

In [ ]:
# Save to processed
import os
os.makedirs('../data/processed', exist_ok=True)
prices.to_csv('../data/processed/prices.csv')
print('Saved prices.csv')

## 2. Closing Price Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
colors = {'TSLA':'#e74c3c', 'BND':'#2ecc71', 'SPY':'#3498db'}
labels = {'TSLA':'Tesla (High Risk)', 'BND':'Vanguard Bond ETF (Low Risk)', 'SPY':'S&P 500 ETF (Moderate Risk)'}

for ax, ticker in zip(axes, TICKERS):
    ax.plot(prices.index, prices[ticker], color=colors[ticker], linewidth=1.2)
    ax.fill_between(prices.index, prices[ticker], alpha=0.08, color=colors[ticker])
    ax.set_ylabel(f'{ticker} Price (USD)', fontsize=11)
    ax.set_title(labels[ticker], fontweight='bold', fontsize=12)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Closing Prices: TSLA, BND, SPY (2015–2026)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('closing_prices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved!')

## 3. Daily Returns & Volatility

In [ ]:
returns = prices.pct_change().dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Daily returns
for ticker in TICKERS:
    axes[0].plot(returns.index, returns[ticker]*100, alpha=0.6,
                 linewidth=0.6, label=ticker, color=colors[ticker])
axes[0].set_title('Daily Returns (%) — All Assets', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Daily Return (%)')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].legend()

# Rolling 30-day volatility
rolling_vol = returns.rolling(30).std() * np.sqrt(252) * 100
for ticker in TICKERS:
    axes[1].plot(rolling_vol.index, rolling_vol[ticker],
                 linewidth=1.2, label=ticker, color=colors[ticker])
axes[1].set_title('Rolling 30-Day Annualized Volatility (%)', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Annualized Volatility (%)')
axes[1].legend()

plt.suptitle('Returns & Volatility Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('returns_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Return Statistics ===')
print(returns.describe().round(4))

In [ ]:
# Return distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, ticker in zip(axes, TICKERS):
    ax.hist(returns[ticker]*100, bins=80, color=colors[ticker],
            edgecolor='white', alpha=0.8, density=True)
    mu = returns[ticker].mean()*100
    ax.axvline(mu, color='black', linestyle='--',
               label=f'Mean: {mu:.3f}%')
    ax.set_title(f'{ticker} Return Distribution', fontweight='bold')
    ax.set_xlabel('Daily Return (%)')
    ax.legend(fontsize=9)

plt.suptitle('Daily Return Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('return_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation & Outlier Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
corr = returns.corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdYlGn',
            center=0, ax=axes[0], square=True, vmin=-1, vmax=1)
axes[0].set_title('Return Correlation Matrix', fontweight='bold')

# Outlier detection — returns beyond 3 std
for ticker in TICKERS:
    mu = returns[ticker].mean()
    std = returns[ticker].std()
    outliers = returns[abs(returns[ticker] - mu) > 3*std][ticker]
    axes[1].scatter(outliers.index, outliers*100, label=f'{ticker} ({len(outliers)} outliers)',
                    alpha=0.7, s=25, color=colors[ticker])
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Outlier Returns (> 3 Std Dev)', fontweight='bold')
axes[1].set_ylabel('Return (%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('correlation_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Correlation Matrix ===')
print(corr.round(3))

## 5. Stationarity Testing (ADF Test)

In [ ]:
def adf_test(series, name):
    result = adfuller(series.dropna())
    stat, pval = result[0], result[1]
    stationary = pval < 0.05
    print(f'{name}:')
    print(f'  ADF Statistic: {stat:.4f}')
    print(f'  p-value: {pval:.6f}')
    print(f'  Result: {"STATIONARY" if stationary else "NON-STATIONARY"} (p {"<" if stationary else ">="} 0.05)')
    print()
    return stationary

print('=== ADF Test on CLOSING PRICES ===')
for ticker in TICKERS:
    adf_test(prices[ticker], ticker)

print('=== ADF Test on DAILY RETURNS ===')
for ticker in TICKERS:
    adf_test(returns[ticker], ticker)

print('Interpretation:')
print('- Closing prices are NON-STATIONARY (expected — prices trend over time)')
print('- Daily returns are STATIONARY (required for ARIMA modeling)')
print('- For ARIMA: d=1 (first-difference the price series to get returns)')

## 6. Risk Metrics: VaR & Sharpe Ratio

In [ ]:
RISK_FREE = 0.05 / 252  # Daily risk-free rate (~5% annual)

print('=== RISK METRICS SUMMARY ===')
print(f'{"Asset":<8} {"Ann.Return":<14} {"Ann.Vol":<12} {"Sharpe":<10} {"VaR 95%":<12} {"VaR 99%"}')
print('-' * 70)

risk_metrics = {}
for ticker in TICKERS:
    r = returns[ticker]
    ann_return = r.mean() * 252
    ann_vol    = r.std() * np.sqrt(252)
    sharpe     = (r.mean() - RISK_FREE) / r.std() * np.sqrt(252)
    var_95     = np.percentile(r, 5)
    var_99     = np.percentile(r, 1)

    risk_metrics[ticker] = {
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe,
        'VaR 95%': var_95,
        'VaR 99%': var_99,
    }
    print(f'{ticker:<8} {ann_return*100:>10.2f}%   {ann_vol*100:>8.2f}%   '
          f'{sharpe:>8.2f}   {var_95*100:>8.2f}%   {var_99*100:.2f}%')

print()
print('VaR 95%: On 95% of days, loss will NOT exceed this percentage')
print('Sharpe:  Risk-adjusted return (higher = better per unit of risk)')

In [ ]:
# Visualize risk metrics
metrics_df = pd.DataFrame(risk_metrics).T * 100

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, color_list in zip(
    axes,
    ['Annual Return', 'Annual Volatility', 'Sharpe Ratio'],
    [[colors[t] for t in TICKERS]]*3
):
    data = metrics_df[col] if col == 'Sharpe Ratio' else metrics_df[col]
    if col == 'Sharpe Ratio':
        data = pd.Series({t: risk_metrics[t]['Sharpe Ratio'] for t in TICKERS})
    axes[list(axes).index(ax)].bar(TICKERS, data, color=color_list, edgecolor='white')
    ax.set_title(col, fontweight='bold')
    ax.set_ylabel('%' if col != 'Sharpe Ratio' else 'Ratio')
    for i, v in enumerate(data):
        ax.text(i, v + (abs(v)*0.03), f'{v:.2f}', ha='center', fontsize=10)

plt.suptitle('Risk Metrics Summary — TSLA, BND, SPY', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('risk_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key EDA Insights

### Finding 1: TSLA dominates in both return and risk
TSLA shows the highest annualized return but also the highest volatility — 3-5x higher than BND and SPY. This confirms its high-risk/high-return profile in the GMF portfolio.

### Finding 2: BND provides stability
BND has near-zero correlation with TSLA, making it an effective hedge. Its VaR is extremely low and Sharpe Ratio is positive despite modest returns.

### Finding 3: Prices are non-stationary, returns are stationary
ADF tests confirm closing prices trend upward (non-stationary), while daily returns are stationary. This means ARIMA requires d=1 (first-differencing) when applied to price series.

### Finding 4: COVID-19 and 2022 market crash visible
TSLA shows extreme outlier returns in March 2020 (COVID crash, -35% in days) and November 2021 (post-earnings spike). These events are visible in both the return distribution tails and the outlier scatter plot.

### Finding 5: SPY and BND correlation is negative
SPY and BND show slight negative correlation — bonds typically rise when stocks fall, confirming the diversification benefit of including BND in the portfolio.